In [2]:
!pip install scikit-fuzzy

import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# 1. Domain variabel
x_suhu = np.arange(0, 101, 1)  # Suhu dari 0 hingga 100 (°C) [cite: 23]
x_tekanan = np.arange(0, 11, 1)  # Tekanan dari 0 hingga 10 (Bar) [cite: 24]

# Fungsi keanggotaan (Suhu) menggunakan trimf (fungsi segitiga)
# Suhu: rendah, sedang, tinggi [cite: 3]
suhu_rendah = fuzz.trimf(x_suhu, [0, 0, 40])   # [cite: 27, 28, 29]
suhu_sedang = fuzz.trimf(x_suhu, [30, 50, 70]) # [cite: 27, 31, 33]
suhu_tinggi = fuzz.trimf(x_suhu, [60, 100, 100]) # [cite: 27, 35, 36]

# Fungsi keanggotaan (Tekanan) menggunakan trimf
# Tekanan: rendah, sedang, tinggi [cite: 4]
tekanan_rendah = fuzz.trimf(x_tekanan, [0, 0, 4])   # [cite: 38, 39, 43]
tekanan_sedang = fuzz.trimf(x_tekanan, [3, 5, 7])   # [cite: 38, 44, 46]
tekanan_tinggi = fuzz.trimf(x_tekanan, [6, 10, 10]) # [cite: 38, 47, 48]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 13.8 MB/s eta 0:00:00


In [3]:
# 2. Fungsi Sugeno
def sugeno(suhu_input, tekanan_input):
    # Fuzzifikasi: Menentukan derajat keanggotaan input

    # Derajat keanggotaan Suhu (μ_s) [cite: 58, 60]
    μ_s_rendah = fuzz.interp_membership(x_suhu, suhu_rendah, suhu_input)
    μ_s_sedang = fuzz.interp_membership(x_suhu, suhu_sedang, suhu_input)
    μ_s_tinggi = fuzz.interp_membership(x_suhu, suhu_tinggi, suhu_input)

    # Derajat keanggotaan Tekanan (μ_t) [cite: 65]
    μ_t_rendah = fuzz.interp_membership(x_tekanan, tekanan_rendah, tekanan_input)
    μ_t_sedang = fuzz.interp_membership(x_tekanan, tekanan_sedang, tekanan_input)
    μ_t_tinggi = fuzz.interp_membership(x_tekanan, tekanan_tinggi, tekanan_input)

    # Rules (Inferensi Sugeno Orde 0)
    # Output Sugeno adalah konstanta (Orde 0) [cite: 5]

    # Rule 1: IF suhu rendah AND tekanan rendah → risiko = 10 [cite: 7, 8, 71, 73]
    w1 = np.fmin(μ_s_rendah, μ_t_rendah) # AND menggunakan min
    z1 = 10

    # Rule 2: IF suhu sedang AND tekanan sedang → risiko = 50 [cite: 9, 10, 77, 79]
    w2 = np.fmin(μ_s_sedang, μ_t_sedang) # AND menggunakan min
    z2 = 50

    # Rule 3: IF suhu tinggi OR tekanan tinggi → risiko = 90 [cite: 11, 12, 84, 85]
    w3 = np.fmax(μ_s_tinggi, μ_t_tinggi) # OR menggunakan max
    z3 = 90

    # Rule 4: IF suhu tinggi AND tekanan rendah → risiko = 60 [cite: 13, 14, 89, 90, 92]
    w4 = np.fmin(μ_s_tinggi, μ_t_rendah) # AND menggunakan min
    z4 = 60

    # Defuzzifikasi: Weighted Average (Sugeno) [cite: 95]
    w = np.array([w1, w2, w3, w4]) # Semua bobot/predikat
    z = np.array([z1, z2, z3, z4]) # Semua output konsekuen

    # Rumus Defuzzifikasi Sugeno: (Σ w_i * z_i) / (Σ w_i)
    result = np.sum(w * z) / np.sum(w)

    return result

In [4]:
# 3. Widget interaktif dan Visualisasi
def interactive_sugeno(suhu, tekanan):
    # Hitung hasil prediksi [cite: 114]
    hasil = sugeno(suhu, tekanan)
    print(f"🔥 Hasil Prediksi Risiko (Sugeno): {hasil:.2f}") # [cite: 117]

    # Visualisasi [cite: 120, 138]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4)) # [cite: 122]

    # Plot Suhu [cite: 125, 127, 130]
    ax1.plot(x_suhu, suhu_rendah, label="Rendah")
    ax1.plot(x_suhu, suhu_sedang, label="Sedang")
    ax1.plot(x_suhu, suhu_tinggi, label="Tinggi")
    ax1.axvline(suhu, color='red', linestyle='--') # Garis vertikal untuk input suhu [cite: 135]
    ax1.set_title("Fungsi Keanggotaan Suhu") # [cite: 136]
    ax1.legend() # [cite: 137]

    # Plot Tekanan [cite: 139]
    ax2.plot(x_tekanan, tekanan_rendah, label="Rendah")
    ax2.plot(x_tekanan, tekanan_sedang, label="Sedang")
    ax2.plot(x_tekanan, tekanan_tinggi, label="Tinggi")
    ax2.axvline(tekanan, color='red', linestyle='--') # Garis vertikal untuk input tekanan [cite: 139]
    ax2.set_title("Fungsi Keanggotaan Tekanan") # [cite: 139]
    ax2.legend() # [cite: 151]

    plt.show() # [cite: 150]

# Jalankan slider interaktif [cite: 154, 156, 158]
interact(
    interactive_sugeno,
    suhu=FloatSlider(min=0, max=100, step=1, value=34), # Nilai awal diubah ke 34 sesuai output [cite: 161, 165]
    tekanan=FloatSlider(min=0, max=10, step=0.5, value=5) # Nilai awal sesuai output [cite: 161, 167]
)

interactive(children=(FloatSlider(value=34.0, description='suhu', step=1.0), FloatSlider(value=5.0, descriptio…

<function __main__.interactive_sugeno(suhu, tekanan)>